In [1]:
import numpy as np
from jax import jit
from jax import lax
from jax import random
import jax
import jax.numpy as jnp

In [2]:
def impure_print_side_effect(x):
  print("Executing function")  # This is a side-effect
  return x

# The side-effects appear during the first run
print ("First call: ", jit(impure_print_side_effect)(4.))

# Subsequent runs with parameters of same type and shape may not show the side-effect
# This is because JAX now invokes a cached compilation of the function
print ("Second call: ", jit(impure_print_side_effect)(5.))

# JAX re-runs the Python function when the type or shape of the argument changes
print ("Third call, different type: ", jit(impure_print_side_effect)(jnp.array([5.])))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Executing function
First call:  4.0
Second call:  5.0
Executing function
Third call, different type:  [5.]


In [3]:
from jax import make_jaxpr

array = jnp.arange(10)

In [4]:
array

Array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

See [`lax.fori_loop`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.fori_loop.html#jax.lax.fori_loop)

In [5]:
lax.fori_loop(0, 10, lambda i, x: x+array[i], 0)

Array(45, dtype=int32)

In [6]:
sum(array)

Array(45, dtype=int32)

In [7]:
def foo(i, x):
    result = x+array[i]
    jax.debug.print("i={i}, x={x}, result={result}", i=i, x=x, result=result)
    return result

In [8]:
lax.fori_loop(0, 10, foo, 0)

i=0, x=0, result=0
i=1, x=0, result=1
i=2, x=1, result=3
i=3, x=3, result=6
i=4, x=6, result=10
i=5, x=10, result=15
i=6, x=15, result=21
i=7, x=21, result=28
i=8, x=28, result=36
i=9, x=36, result=45


Array(45, dtype=int32)

In [9]:
jax_array = jnp.zeros((3,3), dtype=jnp.float32)

# In place update of JAX's array will yield an error!
try:
    jax_array[1, :] = 1.0
except Exception as ex:
    print("Exception", ex)

Exception JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


In [10]:
class Foo:
    def __init__(self, x):
        self.x = x

In [11]:
foo = Foo(1)

In [12]:
def bar(f):
    return f.x + 3

In [13]:
bar(foo)

4

In [14]:
@jit
def bar(f):
    return f.x + 3

In [15]:
try:
    bar(foo)
except Exception as ex:
    print("Exception", ex)

Exception Error interpreting argument to <function bar at 0x7f9316e84a90> as an abstract array. The problematic value is of type <class '__main__.Foo'> and was passed to the function at path f.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.


In [16]:
from functools import partial

@partial(jit, static_argnums=0)
def bar(f):
    return f.x + 3

In [17]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [18]:
foo.x += 1

In [19]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [20]:
class Foo:
    def __init__(self, x):
        self.x = x

    def _tree_flatten(self):
        children = (self.x,)
        aux_data = {}
        return (children, aux_data)

    @classmethod
    def _tree_unflatten(cls, aux_data, children):
        return cls(*children)

from jax import tree_util
tree_util.register_pytree_node(Foo, Foo._tree_flatten, Foo._tree_unflatten)

In [21]:
@jit
def bar(f):
    return f.x + 3

In [22]:
foo = Foo(1)

In [23]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [24]:
foo.x += 1

In [25]:
bar(foo)

Array(5, dtype=int32, weak_type=True)

In [26]:
np.arange(10)[11]

IndexError: index 11 is out of bounds for axis 0 with size 10

In [27]:
jnp.arange(10)[11]

Array(9, dtype=int32)

In [28]:
jnp.arange(10)[10]

Array(9, dtype=int32)

In [33]:
jnp.arange(10).at[11].get(mode="fill", fill_value=jnp.nan)

Array(-2147483648, dtype=int32)

In [32]:
jnp.arange(10.0).at[11].get(mode="fill", fill_value=jnp.nan)

Array(nan, dtype=float32)

In [34]:
np.sum([1, 2, 3])

np.int64(6)

In [49]:
try:
    jnp.sum([1, 2, 3])
except Exception as ex:
    print(ex)

sum requires ndarray or scalar arguments, got <class 'list'> at position 0.


In [36]:
def permissive_sum(x):
    return jnp.sum(jnp.array(x))

In [38]:
x = list(range(10))
permissive_sum(x)

Array(45, dtype=int32)

In [39]:
make_jaxpr(permissive_sum)(x)

{ lambda ; a:i32[] b:i32[] c:i32[] d:i32[] e:i32[] f:i32[] g:i32[] h:i32[] i:i32[]
    j:i32[]. let
    k:i32[] = convert_element_type[new_dtype=int32 weak_type=False] a
    l:i32[1] = broadcast_in_dim k
    m:i32[] = convert_element_type[new_dtype=int32 weak_type=False] b
    n:i32[1] = broadcast_in_dim m
    o:i32[] = convert_element_type[new_dtype=int32 weak_type=False] c
    p:i32[1] = broadcast_in_dim o
    q:i32[] = convert_element_type[new_dtype=int32 weak_type=False] d
    r:i32[1] = broadcast_in_dim q
    s:i32[] = convert_element_type[new_dtype=int32 weak_type=False] e
    t:i32[1] = broadcast_in_dim s
    u:i32[] = convert_element_type[new_dtype=int32 weak_type=False] f
    v:i32[1] = broadcast_in_dim u
    w:i32[] = convert_element_type[new_dtype=int32 weak_type=False] g
    x:i32[1] = broadcast_in_dim w
    y:i32[] = convert_element_type[new_dtype=int32 weak_type=False] h
    z:i32[1] = broadcast_in_dim y
    ba:i32[] = convert_element_type[new_dtype=int32 weak_type=False]

In [41]:
make_jaxpr(jnp.array)(x)

{ lambda ; a:i32[] b:i32[] c:i32[] d:i32[] e:i32[] f:i32[] g:i32[] h:i32[] i:i32[]
    j:i32[]. let
    k:i32[] = convert_element_type[new_dtype=int32 weak_type=False] a
    l:i32[1] = broadcast_in_dim k
    m:i32[] = convert_element_type[new_dtype=int32 weak_type=False] b
    n:i32[1] = broadcast_in_dim m
    o:i32[] = convert_element_type[new_dtype=int32 weak_type=False] c
    p:i32[1] = broadcast_in_dim o
    q:i32[] = convert_element_type[new_dtype=int32 weak_type=False] d
    r:i32[1] = broadcast_in_dim q
    s:i32[] = convert_element_type[new_dtype=int32 weak_type=False] e
    t:i32[1] = broadcast_in_dim s
    u:i32[] = convert_element_type[new_dtype=int32 weak_type=False] f
    v:i32[1] = broadcast_in_dim u
    w:i32[] = convert_element_type[new_dtype=int32 weak_type=False] g
    x:i32[1] = broadcast_in_dim w
    y:i32[] = convert_element_type[new_dtype=int32 weak_type=False] h
    z:i32[1] = broadcast_in_dim y
    ba:i32[] = convert_element_type[new_dtype=int32 weak_type=False]

In [47]:
def nansum(x):
    mask = ~jnp.isnan(x)
    x_without_nans = x[mask]
    return x_without_nans.sum()

In [48]:
x = jnp.array([1, 2, jnp.nan, 3, 4])
print(nansum(x))

10.0


In [52]:
try:
    jax.jit(nansum)(x)
except Exception as ex:
    print(ex)

Array boolean indices must be concrete; got bool[5]

See https://docs.jax.dev/en/latest/errors.html#jax.errors.NonConcreteBooleanIndexError


In [53]:
def nansum_2(x):
    mask = ~jnp.isnan(x)
    return jnp.where(mask, x, 0).sum()


In [54]:
jax.jit(nansum_2)(x)

Array(10., dtype=float32)

In [56]:
x = random.uniform(random.key(0), (1000,), dtype=jnp.float64)
x.dtype

/tmp/ipykernel_2268533/1258726447.py:1: UserWarning: Explicitly requested dtype float64 is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  x = random.uniform(random.key(0), (1000,), dtype=jnp.float64)


dtype('float32')